# VietMedBridge — 00: Tải và kiểm tra ViBioMIR

Chạy trên Google Colab với runtime CPU. Notebook tải hai file Parquet của một
revision cố định, kiểm tra ID/schema và thống kê domain bằng DuckDB có disk spill.
Dữ liệu cùng checkpoint được lưu vào Google Drive; cần cấp quyền mount Drive ở cell đầu.

Dataset có hai cấu hình query và corpus. Split mang tên train là cách đóng gói
trên Hub; query hiện chỉ chứa id/query, chưa có nhãn reference để tính F2.
Giữ nguyên ID chính thức, kể cả khi ID không liên tục hoặc URL trùng nhau.


In [ ]:
import importlib.util
import json
import os
import subprocess
import sys
from pathlib import Path

if not (3, 11) <= sys.version_info[:2] < (3, 14):
    raise RuntimeError(
        "Dùng Python 3.11–3.13. Trong Runtime > Change runtime type, "
        "có thể chọn Runtime Version 2026.07 (Python 3.12)."
    )
if importlib.util.find_spec("google.colab") is None:
    raise RuntimeError("Notebook này được thiết kế cho Google Colab.")

from google.colab import drive
drive.mount("/content/drive")

# Giữ cùng DATA_ROOT trong cả ba notebook.
DATA_ROOT = Path("/content/drive/MyDrive/VietMedBridge/data")
WORK_DIR = Path("/content/vmb_work")  # temp files/spill ở ổ local của Colab
DATA_ROOT.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

REPO_URL = "https://github.com/Platypus27-coder/VietMedBridge.git"
CODE_REVISION = None  # Có thể đặt full commit SHA; mặc định dùng code_lock đã lưu.
CHECKOUT = Path("/content/VietMedBridge")
lock_path = DATA_ROOT / "code_lock.json"
lock = json.loads(lock_path.read_text()) if lock_path.exists() else {}
if lock and lock.get("repo_url") != REPO_URL:
    raise ValueError("DATA_ROOT đang khóa vào một repo khác.")
reference = CODE_REVISION or lock.get("git_commit") or "main"
if not (CHECKOUT / ".git").exists():
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(CHECKOUT)], check=True)
remote = subprocess.check_output(
    ["git", "-C", str(CHECKOUT), "remote", "get-url", "origin"], text=True
).strip()
if remote != REPO_URL:
    raise ValueError("Checkout hiện tại không thuộc repo VietMedBridge.")
subprocess.run(["git", "-C", str(CHECKOUT), "fetch", "--depth", "1", "origin", reference], check=True)
subprocess.run(["git", "-C", str(CHECKOUT), "checkout", "--detach", "FETCH_HEAD"], check=True)
CODE_COMMIT = subprocess.check_output(
    ["git", "-C", str(CHECKOUT), "rev-parse", "HEAD"], text=True
).strip()
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(CHECKOUT) + "[notebook]"],
    check=True,
)
from vietmedbridge.artifacts import atomic_json, runtime_versions
if not lock or CODE_REVISION:
    atomic_json(lock_path, {"repo_url": REPO_URL, "git_commit": CODE_COMMIT})
atomic_json(DATA_ROOT / "runtime.json", {
    "git_commit": CODE_COMMIT, "python": sys.version, "packages": runtime_versions()
})
print("Code commit:", CODE_COMMIT)
print("Dữ liệu/checkpoint:", DATA_ROOT)


## 1. Tải snapshot gắn revision và kiểm tra SHA-256


In [ ]:
from vietmedbridge.dataset import DATASET_REVISION, snapshot_dataset, audit_dataset

# Revision đã kiểm tra của AIGuruTinix/ViBioMIR. Đổi revision sẽ tạo snapshot mới.
SNAPSHOT = snapshot_dataset(DATA_ROOT, revision=DATASET_REVISION)
print("Dataset revision:", SNAPSHOT["revision"])
for name, entry in SNAPSHOT["files"].items():
    print(name, entry.get("rows"), entry["bytes"], entry["sha256"])


## 2. Audit trực tiếp file Parquet

Không coi số dòng trong dataset card là số dòng đã xác minh. Báo cáo ghi số
dòng, số ID duy nhất, ID min/max, URL trùng, domain và độ dài query thực tế.
Sample chọn vài ID đầu của các domain lớn để thử crawler, không dùng làm tập đánh giá.


In [ ]:
REPORT = audit_dataset(DATA_ROOT, work_dir=WORK_DIR, sample_domains=30, per_domain=3)
print(json.dumps({key: REPORT[key] for key in ("query", "corpus", "sample")}, ensure_ascii=False, indent=2))
import pandas as pd
display(pd.DataFrame(REPORT["top_domains"]))


## 3. Cách dùng load_dataset với đúng cấu hình

Query nhỏ được tải bình thường; corpus dùng streaming và chỉ xem ba dòng.
Corpus này là danh sách liên kết, chưa phải nội dung văn bản đã crawl.


In [ ]:
from datasets import load_dataset
from itertools import islice

queries = load_dataset(
    "AIGuruTinix/ViBioMIR", "query", split="train",
    revision=SNAPSHOT["revision"], cache_dir="/content/hf_cache/datasets",
)
corpus_preview = load_dataset(
    "AIGuruTinix/ViBioMIR", "corpus", split="train",
    revision=SNAPSHOT["revision"], streaming=True,
)
print("Queries:", len(queries), queries.features)
display(pd.DataFrame(islice(corpus_preview, 3)))


## Kết quả để dùng ở notebook 01

Google Drive/VietMedBridge/data/raw chứa snapshot nguồn có hash.
reports/dataset_audit.json chứa kết quả audit; reports/domains.parquet chứa
thống kê domain và reports/sample_links.parquet chứa mẫu thử crawler.

Lần đầu code_lock.json lưu commit GitHub đang chạy. Các notebook sau dùng lại
commit này. Khi chủ động nâng code, đặt CODE_REVISION và dùng tên crawl/build run mới.
Nguồn: [ViBioMIR](https://huggingface.co/datasets/AIGuruTinix/ViBioMIR).
